# Phase 1: Structured output with the Triage Agent

**Concept.** A language model normally answers in free text. Our software needs reliable fields such as `category` and `urgency`. So we describe the answer we want as a Pydantic model (a *contract*), ask the model to follow it, and Pydantic checks the result. If the model breaks the contract we get an error, not silent bad data.

**Agent.** This is our first agent, the Triage Agent. It has no tools: it reads one citizen report and decides the language, an English translation, the category, a summary and the urgency.

Run each cell in order with the Run button. Your output may differ slightly from the example in the notes, because language models are not deterministic.

## 1. Load the code and settings

The real logic lives in `src/community_voice/`. This notebook only imports it. 

In [ ]:
import json
from pathlib import Path

import community_voice
from community_voice.agents.triage import triage
from community_voice.core.config import get_settings
from community_voice.schemas.report import ReportIn, SampleReport
from community_voice.schemas.triage import TriageResult

ROOT = Path(community_voice.__file__).resolve().parents[2]
settings = get_settings()
print("Gemini model:", settings.gemini_model)
print("Groq model:  ", settings.groq_model)

## 2. The contract

These are the fields the model must fill in. The descriptions are sent to the model, so they act as instructions.

In [ ]:
for name, field in TriageResult.model_fields.items():
    print(f"{name:16} {field.description}")

## 3. Triage one report

An isiZulu report about a leaking water pipe in Umlazi. We never tell the model the language.

In [ ]:
report = ReportIn(
    text="Amanzi ayaphuma emgwaqeni e-Umlazi V Section, "
    "kuvuza ipayipi elikhulu kusukela izolo."
)
result = triage(report)
print(result.model_dump_json(indent=2))

`result` is a real `TriageResult` object, not text. Try `result.category` and `result.urgency` in a new cell.

## 4. Triage all 10 sample reports

These fictional reports are in English, isiZulu, isiXhosa, Afrikaans and Sesotho. Each has the answer we expect, so we can check the agent.

In [ ]:
path = ROOT / "data" / "sample_reports" / "reports.json"
samples = [
    SampleReport.model_validate(item)
    for item in json.loads(path.read_text(encoding="utf-8"))
]
print(len(samples), "sample reports loaded")

In [ ]:
results = []
for sample in samples:
    result = triage(ReportIn(text=sample.text))
    results.append(result)
    cat_ok = result.category == sample.expected_category
    urg_ok = result.urgency == sample.expected_urgency
    print(
        f"{sample.id}  {result.language.value:5} {result.category.value:12}"
        f" {result.urgency.value:9} category ok: {cat_ok!s:5}"
        f" urgency ok: {urg_ok!s:5}"
    )

In [ ]:
pairs = list(zip(samples, results, strict=True))
category_hits = sum(r.category == s.expected_category for s, r in pairs)
urgency_hits = sum(r.urgency == s.expected_urgency for s, r in pairs)
print(f"Category matches our label: {category_hits}/{len(samples)}")
print(f"Urgency matches our label:  {urgency_hits}/{len(samples)}")

A mismatch is not always the model's fault: urgency is a judgement call. Read the model's reasoning for each report and decide whether you agree.

In [ ]:
for sample, result in pairs:
    print(sample.id, "|", result.summary)
    print("     urgency:", result.urgency.value, "-", result.urgency_reason)

## 5. Swap the model provider

Our provider layer lets an agent use a different model by changing one word. Here we run the same report on Groq. Some models do not support structured output, so if this fails the error is shown instead of stopping the notebook.

In [ ]:
try:
    groq_result = triage(ReportIn(text=samples[1].text), provider="groq")
    print(groq_result.model_dump_json(indent=2))
except Exception as error:
    print("Groq structured output failed:", type(error).__name__, error)

## What to notice

- The model returned typed fields we can rely on, in any of five languages.
- The answers are checked against the contract: an unknown category would raise an error.
- Changing the urgency guide in `src/community_voice/agents/triage.py` changes the model's behaviour. 